[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_13/MFM_ch13_ffd/MFM_ch13_ffd.ipynb)

# MFM_ch13_ffd

**Modelling Financial Markets (MFM) — Chapter 13: Machine Learning in Finance**

Fractional differentiation with a fixed-width window (FFD) of the S&P 500 log price: weights w_k for several orders d, ADF statistic and correlation with the original series as a function of d, and why ADF rejection is not stationarity: the truncated weights sum to 0.178 at d = 0.25, so the FFD series keeps a unit-root component. The memory parameter is estimated directly by local Whittle and exact local Whittle, with 95% confidence bands.

*Author: Daniel Traian Pele*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/MFM/Chapter_13'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import sys
if 'google.colab' in sys.modules:
    !pip -q install shap

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from statsmodels.tsa.stattools import adfuller
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

## Style and data

In [ ]:
# Stil standard MFM (identic cu SFM): transparent + ENG + legenda jos
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Culori brand
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Gray     = '#7F7F7F'
LightGray = '#DADADA'

SEED = 42


def save_fig(name):
    """Salveaza figura ca PDF si PNG transparent, in folderul curent."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Plaseaza legenda in afara graficului, jos-centru."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


In [ ]:
import os
DATA_URL = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
SERIES = {'sp500': 'GSPC.INDX', 'vix': 'VIX.INDX', 'btc': 'BTC-USD.CC'}
START = {'sp500': '2000-01-01', 'vix': '2000-01-01', 'btc': '2014-09-17'}


def load_data(name='sp500', start=None, end='2026-09-18'):
    """OHLCV zilnic din data/market: copia locala, altfel fisierul din repo-ul GitHub MFM."""
    fname = SERIES[name] + '.csv'
    local = [os.path.join(d, 'data', 'market', fname) for d in ('.', '..', '../..', '../../..')]
    path = next((p for p in local if os.path.exists(p)), DATA_URL + fname)
    df = pd.read_csv(path, parse_dates=['date']).set_index('date')
    df = df.rename(columns=str.capitalize)[['Open', 'High', 'Low', 'Close', 'Volume']]
    return df.loc[start or START[name]:end]


spx = load_data('sp500')['Close']
vix = load_data('vix')['Close']
btc = load_data('btc')['Close']
log_spx = np.log(spx)
print('S&P 500:', spx.index[0].date(), '->', spx.index[-1].date(), len(spx), 'obs')
print('BTC    :', btc.index[0].date(), '->', btc.index[-1].date(), len(btc), 'obs')

## Helper functions

In [ ]:
def ffd_weights(d, threshold=1e-4, max_size=10_000):
    """Ponderile w_k = -w_{k-1} (d - k + 1) / k, trunchiate cand |w_k| < threshold."""
    w = [1.0]
    for k in range(1, max_size):
        w_k = -w[-1] * (d - k + 1) / k
        if abs(w_k) < threshold:
            break
        w.append(w_k)
    return np.array(w)


def frac_diff_ffd(series, d, threshold=1e-4):
    """Seria diferentiata fractionar cu fereastra fixa: X~_t = sum_k w_k X_{t-k}."""
    w = ffd_weights(d, threshold)
    width = len(w)
    x = series.dropna().values
    out = np.convolve(x, w, mode='valid')          # out[j] = sum_k w_k x[j+width-1-k]
    return pd.Series(out, index=series.dropna().index[width - 1:])


def find_min_d(logp, grid=np.round(np.arange(0, 1.01, 0.05), 2), crit_level='5%'):
    """Cel mai mic d pentru care seria FFD este stationara (ADF la 5%)."""
    rows = []
    for d in grid:
        x = frac_diff_ffd(logp, d).dropna()
        adf = adfuller(x, maxlag=1, regression='c', autolag=None)
        corr = np.corrcoef(logp.loc[x.index], x)[0, 1]
        rows.append((d, adf[0], adf[4][crit_level], corr))
    out = pd.DataFrame(rows, columns=['d', 'adf', 'crit', 'corr']).set_index('d')
    d_star = out.index[out['adf'] < out['crit']].min()
    return out, d_star


def _periodogram(x, m):
    n = len(x)
    w = np.fft.fft(x)[1:m + 1]
    lam = 2 * np.pi * np.arange(1, m + 1) / n
    return lam, np.abs(w) ** 2 / (2 * np.pi * n)


def local_whittle(x, m=None, alpha=0.65):
    """Estimatorul local Whittle al lui d (Robinson, 1995), cu m = n^alpha frecvente Fourier.
    Intoarce (d_hat, se), se = 1/(2 sqrt(m))."""
    from scipy.optimize import minimize_scalar
    x = np.asarray(x, float)
    m = m or int(len(x) ** alpha)
    lam, I = _periodogram(x - x.mean(), m)
    R = lambda d: np.log(np.mean(lam ** (2 * d) * I)) - 2 * d * np.mean(np.log(lam))
    return minimize_scalar(R, bounds=(-0.49, 1.99), method='bounded').x, 1 / (2 * np.sqrt(m))


def _frac_diff_full(x, d):
    """(1-L)^d x_t cu x_t = 0 pentru t <= 0 (fara trunchiere), prin FFT."""
    n = len(x)
    k = np.arange(1, n)
    pi = np.concatenate([[1.0], np.cumprod((k - 1 - d) / k)])
    L = 2 ** int(np.ceil(np.log2(2 * n)))
    return np.fft.irfft(np.fft.rfft(x, L) * np.fft.rfft(pi, L), L)[:n]


def exact_local_whittle(x, m=None, alpha=0.65):
    """Exact local Whittle (Shimotsu & Phillips, 2005), valid si pentru serii nestationare (d >= 0.5);
    media necunoscuta tratata prin scaderea valorii initiale (Shimotsu, 2010). Intoarce (d_hat, se)."""
    from scipy.optimize import minimize_scalar
    x = np.asarray(x, float)
    x = x - x[0]
    m = m or int(len(x) ** alpha)
    lam = 2 * np.pi * np.arange(1, m + 1) / len(x)
    def R(d):
        _, I = _periodogram(_frac_diff_full(x, d), m)
        return np.log(np.mean(I)) - 2 * d * np.mean(np.log(lam))
    return minimize_scalar(R, bounds=(-0.49, 1.99), method='bounded').x, 1 / (2 * np.sqrt(m))

## Charts

In [ ]:
def fig_ffd_weights():
    fig, ax = plt.subplots(figsize=(6.8, 2.9))
    for d, c in zip([0.2, 0.4, 0.6, 0.8, 1.0], [Forest, Amber, Orange, IDAred, MainBlue]):
        w = ffd_weights(d, threshold=0, max_size=11)
        ax.plot(range(len(w)), w, marker='o', ms=3.5, color=c, label=f'$d={d}$')
    ax.axhline(0, color=Gray, lw=0.5)
    ax.set_xlabel('Lag $k$')
    ax.set_ylabel('Weight $w_k$')
    ax.set_title('Fractional differentiation weights: memory decays slowly for $0<d<1$',
                 fontsize=9, loc='left')
    legend_outside_bottom(ax, ncol=5, y=-0.25)
    plt.tight_layout()
    save_fig('ch13_ffd_weights')

In [ ]:
def fig_ffd_adf():
    out, d_star = find_min_d(log_spx)
    fig, ax = plt.subplots(figsize=(6.8, 3.0))
    ax.plot(out.index, out['adf'], color=MainBlue, marker='o', ms=3, label='ADF statistic')
    ax.axhline(out['crit'].iloc[0], color=IDAred, ls='--', lw=0.9, label='5% critical value')
    ax.axvline(d_star, color=Gray, ls=':', lw=0.9)
    ax.set_xlabel('Differentiation order $d$')
    ax.set_ylabel('ADF statistic')
    ax2 = ax.twinx()
    ax2.plot(out.index, out['corr'], color=Forest, marker='s', ms=3, label='Corr. with log price')
    ax2.set_ylabel('Correlation', color=Forest)
    ax2.spines['right'].set_visible(True)
    ax2.set_ylim(-0.05, 1.05)
    ax.annotate(f"$d^*={d_star:.2f}$\ncorr $={out.loc[d_star, 'corr']:.2f}$",
                xy=(d_star, out.loc[d_star, 'adf']), xytext=(d_star + 0.12, out['adf'].min() * 0.45),
                fontsize=8, arrowprops=dict(arrowstyle='->', color=Gray, lw=0.6))
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, loc='upper center', bbox_to_anchor=(0.5, -0.22), ncol=3, frameon=False)
    ax.set_title('S&P 500 log price, 2000-2026: ADF (constant, 1 lag) on the FFD series', fontsize=9, loc='left')
    plt.tight_layout()
    save_fig('ch13_ffd_adf')
    return d_star

In [ ]:
def fig_ffd_series(d_star):
    x_ffd = frac_diff_ffd(log_spx, d_star)
    x_d1 = log_spx.diff().dropna()
    fig, axes = plt.subplots(3, 1, figsize=(7.0, 4.2), sharex=True)
    panels = [(log_spx, MainBlue, '$d=0$: log price (memory, non-stationary)'),
              (x_ffd, Forest, f'$d={d_star:.2f}$: FFD series (ADF rejects, yet still I(1): weights sum to {ffd_weights(d_star).sum():.3f})'),
              (x_d1, IDAred, '$d=1$: log returns (stationary, memory erased)')]
    for ax, (s, c, t) in zip(axes, panels):
        ax.plot(s.index, s.values, color=c, lw=0.6)
        ax.set_title(t, fontsize=8.5, loc='left', color=c)
    axes[-1].set_xlabel('Date')
    plt.tight_layout()
    save_fig('ch13_ffd_series')

In [ ]:
def fig_memory_estimation(d_ffd=0.25):
    r = log_spx.diff().dropna()
    x_ffd = frac_diff_ffd(log_spx, d_ffd)
    alphas = np.round(np.arange(0.45, 0.801, 0.025), 3)
    series = [('Log price (exact local Whittle)', log_spx.values, exact_local_whittle, MainBlue),
              (f'FFD series, $d={d_ffd}$ (exact local Whittle)', x_ffd.values, exact_local_whittle, Forest),
              ('Absolute returns $|r_t|$ (local Whittle)', r.abs().values, local_whittle, Amber),
              ('Returns $r_t$ (local Whittle)', r.values, local_whittle, IDAred)]
    fig, ax = plt.subplots(figsize=(6.8, 3.0))
    out = {}
    for lab, x, est, c in series:
        res = np.array([est(x, alpha=a) for a in alphas])
        ax.plot(alphas, res[:, 0], color=c, marker='o', ms=2.5, label=lab)
        ax.fill_between(alphas, res[:, 0] - 1.96 * res[:, 1], res[:, 0] + 1.96 * res[:, 1], color=c, alpha=0.15, lw=0)
        out[lab] = dict(zip(alphas, res[:, 0]))
    for v in (0, 0.5, 1):
        ax.axhline(v, color=Gray, ls=':', lw=0.7)
    ax.axvline(0.65, color=Gray, ls='--', lw=0.7)
    ax.set_xlabel('Bandwidth exponent $a$ (number of frequencies $m = n^{a}$)')
    ax.set_ylabel('Estimated memory $\\hat d$')
    ax.set_title('S&P 500, 2000-2026: memory parameter with 95% CI ($\\pm 1.96/(2\\sqrt{m})$)', fontsize=9, loc='left')
    legend_outside_bottom(ax, ncol=2, y=-0.25)
    plt.tight_layout()
    save_fig('ch13_memory_estimation')
    return out

## Run

In [ ]:
fig_ffd_weights()
d_star = fig_ffd_adf()
print('d* =', d_star, '| sum of FFD weights =', round(ffd_weights(d_star).sum(), 3))
fig_ffd_series(d_star)
fig_memory_estimation(d_star)

![ch13_ffd_weights](./ch13_ffd_weights.png)

![ch13_ffd_adf](./ch13_ffd_adf.png)

![ch13_ffd_series](./ch13_ffd_series.png)

![ch13_memory_estimation](./ch13_memory_estimation.png)

Output: `ch13_ffd_weights.pdf`, `ch13_ffd_adf.pdf`, `ch13_ffd_series.pdf`, `ch13_memory_estimation.pdf`